# 3D U-Net Inference & Testing

This notebook is for:
1. Loading a trained model
2. Testing on custom OBJ files from a directory
3. Displaying metrics and saving results

## Memory Requirements
- **Light model** (depth=3, features=16): ~2GB GPU
- **Standard model** (depth=4, features=32): ~8GB GPU
- If you have 12GB GPU, use the light model or reduce batch size to 1

In [ ]:

import os, sys, json
import numpy as np
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Check GPU and memory
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total Memory: {total_mem:.1f} GB")
    
    # Clear any cached memory
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f"Currently Allocated: {allocated:.2f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4070
Total Memory: 12.9 GB
Currently Allocated: 0.10 GB


In [12]:
from models.unet3d import get_model
from data.mesh_utils import (
    load_mesh, mesh_to_voxels, voxels_to_mesh, 
    normalize_mesh, prepare_multi_view_input
)
from metrics import compute_all_metrics, MetricsCalculator

print("Imports OK")

Imports OK


## Configuration

In [ ]:
# ===== PATHS =====
CHECKPOINT_PATH = "./outputs/.../best.pt"  # <-- CHANGE THIS

# ===== TEST MODE =====
# 1 = Individual   : TEST_OBJ_DIR is a folder for ONE sample.
#                    .obj files are found recursively inside it.
# 2 = Multiple     : TEST_OBJ_DIR contains many sub-folders, one per sample.
#                    .obj files are discovered recursively (any depth).
# 3 = Random combos: TEST_OBJ_DIR is a single sample folder with layout
#                    view_{angle}/0/mesh.obj  (one .obj per view sub-folder).
#                    N_COMBINATIONS random subsets of N_VIEWS views are tested.
TEST_MODE = 3  # <-- CHANGE THIS (1 / 2 / 3)

# How many random view-combinations to test (Mode 3 only)
N_COMBINATIONS = 50  # <-- CHANGE THIS

# Path to test directory (meaning depends on TEST_MODE - see above)
TEST_OBJ_DIR ="./New_Result/.../000001_gvxr_processed"   # <-- CHANGE THIS
FOLDER_DIR = TEST_OBJ_DIR.replace("/", "_")

# Path to ground truth STL file (optional, for metrics)
GROUND_TRUTH_PATH = "./Bone_V5.stl"  # <-- CHANGE THIS (or None if no GT)

# Output directory for results
OUTPUT_DIR = "./test_results"

# ===== SCENE ROTATION =====
# If X-ray images were captured after gvxr.rotateScene(rx, ry, rz),
# set the same angles here so the GT mesh is rotated into the model's
# coordinate frame before metrics are computed.
# Set to None if no rotation was applied.
SCENE_ROTATION = (0, 0, 0)  # (rx, ry, rz) degrees - or None

# ===== VIEW ANGLE FILTERING (Mode 3 only) =====
# Restrict random combinations to views whose angle is in this list,
# matching the valid angles used during training.
# Set to None to allow all available views.
VALID_ANGLES = list(range(-180, -148)) + list(range(-20, -10)) + list(range(10, 20))+list(range(145, 150))
#VALID_ANGLES = None  # uncomment to disable filtering

# Minimum circular angular separation (degrees) between any two chosen
# views in a combination. Set to 0 to disable.
MIN_ANGLE_SEPARATION = 30  # <-- CHANGE THIS

# ===== MODEL CONFIG (must match checkpoint!) =====
N_VIEWS = 3              # Number of input views
RESOLUTION = 192         # Voxel resolution
MODEL_TYPE = "attention" # "standard", "attention", "multiscale"

# ===== MEMORY-SAVING OPTIONS =====
USE_LIGHT_MODEL = True
BASE_FEATURES = 16
DEPTH = 4
BATCH_SIZE = 1
USE_FP16 = False

MODE_LABELS = {1: 'individual', 2: 'multiple', 3: 'random-combos'}
print(f"Config: {N_VIEWS} views, {RESOLUTION}^3, depth={DEPTH}, features={BASE_FEATURES}")
print(f"Light model: {USE_LIGHT_MODEL}, FP16: {USE_FP16}")
print(f"Test mode : {TEST_MODE} ({MODE_LABELS[TEST_MODE]})")
if TEST_MODE == 3:
    print(f"  Combinations      : {N_COMBINATIONS}")
    print(f"  Min angle gap     : {MIN_ANGLE_SEPARATION} deg")
    n_valid = len(VALID_ANGLES) if VALID_ANGLES is not None else 'all'
    print(f"  Valid angles      : {n_valid}")
if SCENE_ROTATION:
    print(f"Scene rotation: rx={SCENE_ROTATION[0]} deg, ry={SCENE_ROTATION[1]} deg, rz={SCENE_ROTATION[2]} deg")
else:
    print("Scene rotation: none")


Config: 3 views, 192^3, depth=4, features=16
Light model: True, FP16: False
Test mode : 3 (random-combos)
  Combinations      : 50
  Min angle gap     : 30 deg
  Valid angles      : 57
Scene rotation: rx=0 deg, ry=0 deg, rz=0 deg


## Estimate Model Size

In [ ]:
def estimate_model_memory(in_channels, base_features, depth, resolution):
    """Estimate GPU memory needed for model + one forward pass."""
    # Rough parameter count estimate
    features = [base_features * (2 ** i) for i in range(depth + 1)]
    params = 0
    
    # Encoder + Decoder convolutions (rough estimate)
    for i in range(depth):
        params += features[i] * features[i+1] * 27 * 2  # 3x3x3 conv
        params += features[i+1] * features[i] * 27 * 2
    
    # Bottleneck
    params += features[-1] * features[-1] * 2 * 27 * 2
    
    param_memory_mb = params * 4 / 1e6  # float32
    
    # Activation memory (rough estimate for one forward pass)
    activation_memory_mb = 0
    res = resolution
    for i in range(depth + 1):
        activation_memory_mb += features[min(i, len(features)-1)] * res**3 * 4 / 1e6
        res = res // 2
    activation_memory_mb *= 2  # Forward + backward
    
    total_gb = (param_memory_mb + activation_memory_mb) / 1000
    
    return {
        'parameters': params,
        'param_memory_mb': param_memory_mb,
        'activation_memory_mb': activation_memory_mb,
        'total_estimated_gb': total_gb
    }

# Estimate for current config
est = estimate_model_memory(N_VIEWS, BASE_FEATURES, DEPTH, RESOLUTION)
print(f"Estimated parameters: {est['parameters']:,}")
print(f"Estimated memory: {est['total_estimated_gb']:.1f} GB")

if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    if est['total_estimated_gb'] > total_mem * 0.8:
        print(f"\n WARNING: Model may exceed GPU memory ({total_mem:.1f} GB)!")
        print("   Try: USE_LIGHT_MODEL=True, or reduce RESOLUTION to 48")
    else:
        print(f"\n Should fit in GPU memory ({total_mem:.1f} GB)")

Estimated parameters: 11,778,048
Estimated memory: 1.3 GB

[OK] Should fit in GPU memory (12.9 GB)


## Load Trained Model

In [ ]:
# Clear GPU cache first
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Create model
model = get_model(
    model_type=MODEL_TYPE,
    in_channels=N_VIEWS,
    out_channels=1,
    base_features=BASE_FEATURES,
    depth=DEPTH
)

# Count actual parameters
n_params = sum(p.numel() for p in model.parameters())
param_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
print(f"Model parameters: {n_params:,}")
print(f"Model size: {param_size_mb:.1f} MB")

# Load checkpoint
if Path(CHECKPOINT_PATH).exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')  # Load to CPU first
    
    # Check if config matches
    saved_config = checkpoint.get('config', {})
    if saved_config.get('n_views') and saved_config['n_views'] != N_VIEWS:
        print(f"  WARNING: Checkpoint was trained with {saved_config['n_views']} views, but N_VIEWS={N_VIEWS}")
    
    model.load_state_dict(checkpoint['model'])
    print(f"\n[OK] Loaded checkpoint from: {CHECKPOINT_PATH}")
    print(f"  Trained for {checkpoint.get('epoch', '?')+1} epochs")
    if 'val_metrics' in checkpoint:
        print(f"  Val Dice: {checkpoint['val_metrics'].get('dice', '?'):.4f}")
else:
    print(f"  Checkpoint not found: {CHECKPOINT_PATH}")
    print("   Using randomly initialized model (for testing only)")

# Move to device
model = model.to(device)
model.eval()

# Use FP16 if enabled
if USE_FP16 and torch.cuda.is_available():
    model = model.half()
    print("Using FP16 (half precision)")

print(f"\nModel on device: {next(model.parameters()).device}")

Model parameters: 23,097,829
Model size: 92.4 MB

[OK] Loaded checkpoint from: ./outputs/20260113_0044_v3_r192_E0.06/best.pt
  Trained for 16 epochs
  Val Dice: 0.9108

Model on device: cuda:0


## Prepare Test Data from Directory

Put your 3 (or N_VIEWS) OBJ files in a directory. The code will load them as multi-view input.

In [ ]:
import random
from itertools import combinations
from scipy.spatial.transform import Rotation as ScipyRotation


def apply_scene_rotation(mesh, angles_xyz_deg):
    """
    Rotate *mesh* by XYZ Euler angles matching gvxr.rotateScene(rx, ry, rz).
    Applied about the mesh centroid, in degrees.
    """
    rx, ry, rz = angles_xyz_deg
    rot = ScipyRotation.from_euler('xyz', [rx, ry, rz], degrees=True)
    mesh = mesh.copy()
    centroid = mesh.centroid
    mesh.vertices = rot.apply(mesh.vertices - centroid) + centroid
    return mesh


class TestDataset(Dataset):
    """
    Dataset for testing with OBJ files - three modes:

    Mode 1 - Individual
        test_dir/ is a single sample folder.
        .obj files collected with rglob; first N_VIEWS (sorted) used.

    Mode 2 - Multiple
        test_dir/ contains one sub-folder per sample.
        .obj files found recursively at any depth; N_VIEWS taken each.

    Mode 3 - Random view combinations
        test_dir/ is a single sample folder with layout:
            view_{angle}/0/mesh.obj  (one .obj per view sub-folder)
        Views are optionally filtered to valid_angles, then
        N_COMBINATIONS unique random subsets of N_VIEWS are drawn while
        enforcing a minimum circular angular separation between all
        views in each combination.

    scene_rotation
        When (rx, ry, rz) is provided, the GT mesh is rotated before
        voxelisation to align with the X-ray capture orientation,
        matching gvxr.rotateScene().
    """

    def __init__(self, test_dir, gt_path=None, n_views=3, resolution=64,
                 mode=1, n_combinations=5, scene_rotation=None,
                 valid_angles=None, min_angle_separation=0):
        self.test_dir             = Path(test_dir)
        self.gt_path              = Path(gt_path) if gt_path else None
        self.n_views              = n_views
        self.resolution           = resolution
        self.mode                 = mode
        self.n_combinations       = n_combinations
        self.scene_rotation       = scene_rotation
        self.valid_angles         = set(valid_angles) if valid_angles is not None else None
        self.min_angle_separation = min_angle_separation

        self.samples = self._find_samples()
        print(f"Mode {mode} - found {len(self.samples)} test sample(s)")

    # ------------------------------------------------------------------ #
    #  Helpers                                                             #
    # ------------------------------------------------------------------ #

    def _find_obj_files_recursive(self, folder):
        """All .obj files under *folder*, sorted by full path string."""
        return sorted(Path(folder).rglob('*.obj'), key=lambda p: str(p))

    @staticmethod
    def _parse_angle(view_dir_name):
        """Parse integer angle from 'view_+30' or 'view_-141'. Returns None on failure."""
        try:
            return int(view_dir_name.replace('view_', '').lstrip('+'))
        except ValueError:
            return None

    @staticmethod
    def _angular_distance(a1, a2):
        """Circular angular distance in degrees (0-180)."""
        diff = abs(a1 - a2) % 360
        return min(diff, 360 - diff)

    def _combo_is_valid(self, angles):
        """True if every pair of angles meets the minimum separation."""
        if self.min_angle_separation <= 0:
            return True
        for i in range(len(angles)):
            for j in range(i + 1, len(angles)):
                if self._angular_distance(angles[i], angles[j]) < self.min_angle_separation:
                    return False
        return True

    # ------------------------------------------------------------------ #
    #  Sample discovery                                                    #
    # ------------------------------------------------------------------ #

    def _find_samples(self):
        if self.mode == 1:
            return self._mode_individual()
        elif self.mode == 2:
            return self._mode_multiple()
        elif self.mode == 3:
            return self._mode_random_combos()
        raise ValueError(f"Unknown TEST_MODE={self.mode}. Choose 1, 2 or 3.")

    # -- Mode 1 -------------------------------------------------------- #
    def _mode_individual(self):
        obj_files = self._find_obj_files_recursive(self.test_dir)
        if len(obj_files) < self.n_views:
            print(f"  Only {len(obj_files)} .obj file(s) found "
                  f"(need {self.n_views}) in {self.test_dir}")
            return []
        return [{'name': self.test_dir.name,
                 'obj_files': obj_files[:self.n_views]}]

    # -- Mode 2 -------------------------------------------------------- #
    def _mode_multiple(self):
        samples = []
        for subdir in sorted(d for d in self.test_dir.iterdir() if d.is_dir()):
            obj_files = self._find_obj_files_recursive(subdir)
            if len(obj_files) < self.n_views:
                print(f"  Skipping '{subdir.name}': "
                      f"only {len(obj_files)} .obj (need {self.n_views})")
                continue
            if len(obj_files) > self.n_views:
                print(f" '{subdir.name}': {len(obj_files)} .obj files, "
                      f"using first {self.n_views}")
            samples.append({'name': subdir.name,
                            'obj_files': obj_files[:self.n_views]})
        return samples

    # -- Mode 3 -------------------------------------------------------- #
    def _mode_random_combos(self):
        """
        Pick N_COMBINATIONS random view subsets from test_dir.

        Layout:  test_dir/view_{angle}/0/mesh.obj

        Steps
        -----
        1. Collect all view folders and parse their angles.
        2. Filter to VALID_ANGLES (if set).
        3. Enumerate all C(n, N_VIEWS) combos satisfying MIN_ANGLE_SEPARATION.
        4. Draw N_COMBINATIONS unique combos at random.
        """
        # Step 1 - collect
        view_entries = []   # [(angle, Path-to-mesh.obj), ...]
        for vd in sorted(self.test_dir.iterdir(), key=lambda p: str(p)):
            if not vd.is_dir():
                continue
            angle = self._parse_angle(vd.name)
            if angle is None:
                continue
            found = sorted(vd.rglob('mesh.obj'), key=lambda p: str(p))
            if found:
                view_entries.append((angle, found[0]))

        print(f"  Discovered {len(view_entries)} view(s) in '{self.test_dir.name}'")

        # Step 2 - filter to valid angles
        if self.valid_angles is not None:
            before = len(view_entries)
            view_entries = [(a, p) for a, p in view_entries if a in self.valid_angles]
            print(f" After valid-angle filter: {len(view_entries)} / {before} views kept")

        if len(view_entries) < self.n_views:
            print(f" Need at least {self.n_views} views after filtering, "
                  f"found {len(view_entries)}")
            return []

        # Step 3 - enumerate combos that pass the separation constraint
        all_indices  = list(range(len(view_entries)))
        valid_combos = [
            combo for combo in combinations(all_indices, self.n_views)
            if self._combo_is_valid([view_entries[i][0] for i in combo])
        ]
        print(f"  Valid combos (gap >= {self.min_angle_separation} deg): {len(valid_combos):,}")

        if not valid_combos:
            print(f"  No combos satisfy MIN_ANGLE_SEPARATION={self.min_angle_separation} deg. "
                  f"Try lowering it.")
            return []

        # Step 4 - random draw
        n_to_draw = min(self.n_combinations, len(valid_combos))
        if n_to_draw < self.n_combinations:
            print(f" Only {len(valid_combos)} valid combo(s) exist; "
                  f"drawing all {len(valid_combos)}")
        chosen = random.sample(valid_combos, k=n_to_draw)

        samples = []
        for ci, indices in enumerate(chosen):
            angles    = [view_entries[i][0] for i in indices]
            obj_files = [view_entries[i][1] for i in indices]
            angle_strs = [f"view_{'+' if a >= 0 else ''}{a}" for a in angles]
            label = f"combo_{ci+1:02d}__{'+'.join(angle_strs)}"
            samples.append({'name': label, 'obj_files': obj_files})
            print(f"  Combo {ci+1:>3}: angles = {angles}")

        return samples

    # ------------------------------------------------------------------ #
    #  Dataset interface                                                   #
    # ------------------------------------------------------------------ #

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # -- Input views ------------------------------------------------ #
        inputs = prepare_multi_view_input(
            sample['obj_files'],
            resolution=self.resolution,
            align=True,
            fill_interior=True
        )

        # -- Ground truth ----------------------------------------------- #
        target  = None
        gt_info = {}
        if self.gt_path and self.gt_path.exists():
            gt_mesh = load_mesh(self.gt_path)

            # Rotate GT to match the coordinate frame of the X-ray views
            if self.scene_rotation is not None:
                gt_mesh = apply_scene_rotation(gt_mesh, self.scene_rotation)
                gt_info['scene_rotation_deg'] = list(self.scene_rotation)

            gt_info['bounds'] = gt_mesh.bounds.tolist()
            gt_info['center'] = gt_mesh.centroid.tolist()

            gt_mesh_norm, norm_params = normalize_mesh(gt_mesh, return_params=True)
            target = mesh_to_voxels(gt_mesh_norm,
                                    resolution=self.resolution,
                                    fill_interior=True)
            gt_info['scale'] = norm_params.scale

        inputs = torch.from_numpy(inputs.astype(np.float32))
        if target is not None:
            target = torch.from_numpy(target[np.newaxis].astype(np.float32))

        return inputs, target, {
            'name':      sample['name'],
            'obj_files': [str(f) for f in sample['obj_files']],
            'gt_info':   gt_info
        }


In [33]:
# Create test dataset
test_dataset = TestDataset(
    test_dir=TEST_OBJ_DIR,
    gt_path=GROUND_TRUTH_PATH,
    n_views=N_VIEWS,
    resolution=RESOLUTION,
    mode=TEST_MODE,
    n_combinations=N_COMBINATIONS,
    scene_rotation=SCENE_ROTATION,
    valid_angles=VALID_ANGLES,
    min_angle_separation=MIN_ANGLE_SEPARATION,
)

# Show found samples
print(f"\nTest samples:")
for i, sample in enumerate(test_dataset.samples):
    print(f"  [{i}] {sample['name']}")
    for f in sample['obj_files']:
        print(f"      - {f}")


  Discovered 70 view(s) in '000001_gvxr_processed'
  After valid-angle filter: 57 / 70 views kept
  Valid combos (gap >= 30 deg): 5,310
  Combo   1: angles = [149, 16, -161]
  Combo   2: angles = [14, 148, -157]
  Combo   3: angles = [146, -14, -164]
  Combo   4: angles = [14, -171, -20]
  Combo   5: angles = [15, -160, -20]
  Combo   6: angles = [149, -175, -18]
  Combo   7: angles = [149, -167, -18]
  Combo   8: angles = [145, -12, -173]
  Combo   9: angles = [149, 15, -164]
  Combo  10: angles = [148, 16, -19]
  Combo  11: angles = [14, -156, -17]
  Combo  12: angles = [14, -154, -20]
  Combo  13: angles = [14, -174, -20]
  Combo  14: angles = [18, -18, -180]
  Combo  15: angles = [16, -17, -179]
  Combo  16: angles = [19, -15, -178]
  Combo  17: angles = [146, -13, -169]
  Combo  18: angles = [148, -14, -166]
  Combo  19: angles = [148, -14, -170]
  Combo  20: angles = [18, -16, -169]
  Combo  21: angles = [149, -14, -156]
  Combo  22: angles = [148, 15, -167]
  Combo  23: angles =

## Run Inference and Compute Metrics

In [34]:
def run_inference(model, inputs, device, use_fp16=False):
    """Run inference with memory optimization."""
    model.eval()
    
    with torch.no_grad():
        if inputs.dim() == 4:
            inputs = inputs.unsqueeze(0)
        
        inputs = inputs.to(device)
        if use_fp16:
            inputs = inputs.half()
        
        outputs = model(inputs)
        
        # Move back to CPU and convert to float32
        outputs = outputs.float().cpu()
        
    # Clear cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return outputs

In [35]:
# Run inference on all test samples
results = []

print("Running inference...\n")

for i in range(len(test_dataset)):
    inputs, target, metadata = test_dataset[i]
    
    print(f"Sample {i+1}/{len(test_dataset)}: {metadata['name']}")
    
    # Run inference
    pred = run_inference(model, inputs, device, USE_FP16)
    pred = pred[0, 0].numpy()  # (D, H, W)
    
    # Compute metrics if GT available
    metrics = {}
    if target is not None:
        target_np = target[0].numpy()
        metrics = compute_all_metrics(
            torch.from_numpy(pred).unsqueeze(0).unsqueeze(0),
            torch.from_numpy(target_np).unsqueeze(0)
        )
        print(f"  Dice: {metrics['dice']:.4f}")
        print(f"  IoU: {metrics['iou']:.4f}")
        print(f"  PSNR: {metrics['psnr']:.2f} dB")
        print(f"  SSIM: {metrics['ssim']:.4f}")
        print(f"  Chamfer: {metrics['chamfer_distance']:.6f}")
    else:
        print("  (No ground truth - metrics not computed)")
    
    results.append({
        'name': metadata['name'],
        'obj_files': metadata['obj_files'],
        'prediction': pred,
        'target': target[0].numpy() if target is not None else None,
        'metrics': metrics,
        'gt_info': metadata['gt_info']
    })
    print()

Running inference...

Sample 1/50: combo_01__view_+149+view_+16+view_-161
  Dice: 0.9364
  IoU: 0.8804
  PSNR: 20.31 dB
  SSIM: 0.8694
  Chamfer: 0.001516

Sample 2/50: combo_02__view_+14+view_+148+view_-157
  Dice: 0.9394
  IoU: 0.8857
  PSNR: 20.39 dB
  SSIM: 0.8716
  Chamfer: 0.001728

Sample 3/50: combo_03__view_+146+view_-14+view_-164
  Dice: 0.9419
  IoU: 0.8901
  PSNR: 20.55 dB
  SSIM: 0.8710
  Chamfer: 0.001485

Sample 4/50: combo_04__view_+14+view_-171+view_-20
  Dice: 0.9092
  IoU: 0.8335
  PSNR: 18.70 dB
  SSIM: 0.8615
  Chamfer: 0.001714

Sample 5/50: combo_05__view_+15+view_-160+view_-20
  Dice: 0.9399
  IoU: 0.8867
  PSNR: 20.57 dB
  SSIM: 0.8711
  Chamfer: 0.001470

Sample 6/50: combo_06__view_+149+view_-175+view_-18
  Dice: 0.6516
  IoU: 0.4832
  PSNR: 13.25 dB
  SSIM: 0.8056
  Chamfer: 0.015059

Sample 7/50: combo_07__view_+149+view_-167+view_-18
  Dice: 0.6385
  IoU: 0.4689
  PSNR: 13.33 dB
  SSIM: 0.8136
  Chamfer: 0.018550

Sample 8/50: combo_08__view_+145+view_-12+

## Display Metrics Summary

In [36]:
import csv

# Summary table
print("=" * 80)
print("METRICS SUMMARY")
print("=" * 80)
print(f"{'Sample':<20} {'Dice':>8} {'IoU':>8} {'PSNR':>8} {'SSIM':>8} {'Chamfer':>10}")
print("-" * 80)

valid_results = [r for r in results if r['metrics']]

for r in results:
    m = r['metrics']
    if m:
        print(f"{r['name']:<20} {m['dice']:>8.4f} {m['iou']:>8.4f} {m['psnr']:>8.2f} {m['ssim']:>8.4f} {m['chamfer_distance']:>10.6f}")
    else:
        print(f"{r['name']:<20} {'N/A':>8} {'N/A':>8} {'N/A':>8} {'N/A':>8} {'N/A':>10}")

# Average metrics
if valid_results:
    print("-" * 80)
    avg_dice    = np.mean([r['metrics']['dice']             for r in valid_results])
    avg_iou     = np.mean([r['metrics']['iou']              for r in valid_results])
    avg_psnr    = np.mean([r['metrics']['psnr']             for r in valid_results])
    avg_ssim    = np.mean([r['metrics']['ssim']             for r in valid_results])
    avg_chamfer = np.mean([r['metrics']['chamfer_distance'] for r in valid_results])
    print(f"{'AVERAGE':<20} {avg_dice:>8.4f} {avg_iou:>8.4f} {avg_psnr:>8.2f} {avg_ssim:>8.4f} {avg_chamfer:>10.6f}")

print("=" * 80)

# -- Save as CSV ------------------------------------------------------------ #
csv_path = Path(OUTPUT_DIR) / str(FOLDER_DIR +'.csv')
csv_path.parent.mkdir(parents=True, exist_ok=True)

with open(csv_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['sample', 'dice', 'iou', 'psnr', 'ssim', 'chamfer_distance'])
    for r in results:
        m = r['metrics']
        if m:
            writer.writerow([
                r['name'],
                f"{m['dice']:.6f}",
                f"{m['iou']:.6f}",
                f"{m['psnr']:.6f}",
                f"{m['ssim']:.6f}",
                f"{m['chamfer_distance']:.6f}",
            ])
        else:
            writer.writerow([r['name'], 'N/A', 'N/A', 'N/A', 'N/A', 'N/A'])
    if valid_results:
        writer.writerow(['AVERAGE',
                         f"{avg_dice:.6f}",
                         f"{avg_iou:.6f}",
                         f"{avg_psnr:.6f}",
                         f"{avg_ssim:.6f}",
                         f"{avg_chamfer:.6f}"])

print(f"\n[OK] Metrics CSV saved to: {csv_path}")


METRICS SUMMARY
Sample                   Dice      IoU     PSNR     SSIM    Chamfer
--------------------------------------------------------------------------------
combo_01__view_+149+view_+16+view_-161   0.9364   0.8804    20.31   0.8694   0.001516
combo_02__view_+14+view_+148+view_-157   0.9394   0.8857    20.39   0.8716   0.001728
combo_03__view_+146+view_-14+view_-164   0.9419   0.8901    20.55   0.8710   0.001485
combo_04__view_+14+view_-171+view_-20   0.9092   0.8335    18.70   0.8615   0.001714
combo_05__view_+15+view_-160+view_-20   0.9399   0.8867    20.57   0.8711   0.001470
combo_06__view_+149+view_-175+view_-18   0.6516   0.4832    13.25   0.8056   0.015059
combo_07__view_+149+view_-167+view_-18   0.6385   0.4689    13.33   0.8136   0.018550
combo_08__view_+145+view_-12+view_-173   0.8810   0.7874    17.49   0.8481   0.002242
combo_09__view_+149+view_+15+view_-164   0.9249   0.8603    19.57   0.8659   0.001663
combo_10__view_+148+view_+16+view_-19   0.9583   0.9200    22.0